EPL Power Rankings

Objective

Develop a Premier League Power Rankings model using:

-Historical Match Results
-Goals Scored
-Goals Conceded
-Shot Volume
-Current Season Expected Goals (xG)
-Recent Form

Version 1 Scope

Included:

-Historical Match Results
-Current Form Ratings
-Power Rankings

Excluded:

-Match Predictions
-Home Advantage
-Betting Models
-Season Simulations

Data Source

-football-data.co.uk

Imports

In [2]:
import pandas as pd
import numpy as np

In [3]:
print("Imports Loaded")

Imports Loaded


Data Acquisition

In [5]:
season_2024_25 = pd.read_csv(
    "https://www.football-data.co.uk/mmz4281/2425/E0.csv"
)

season_2025_26 = pd.read_csv(
    "https://www.football-data.co.uk/mmz4281/2526/E0.csv"
)

season_2026_27 = pd.read_csv(
    "https://www.football-data.co.uk/mmz4281/2627/E0.csv"
)

In [6]:
print("2024/25:", season_2024_25.shape)

print("2025/26:", season_2025_26.shape)

print("2026/27:", season_2026_27.shape)

2024/25: (380, 120)
2025/26: (380, 132)
2026/27: (60, 114)


In [7]:
season_2026_27.columns.tolist()

['Div',
 'Date',
 'Time',
 'HomeTeam',
 'AwayTeam',
 'FTHG',
 'FTAG',
 'FTR',
 'HTHG',
 'HTAG',
 'HTR',
 'Referee',
 'HxG',
 'AxG',
 'HS',
 'AS',
 'HST',
 'AST',
 'HF',
 'AF',
 'HC',
 'AC',
 'HY',
 'AY',
 'HR',
 'AR',
 'B365H',
 'B365D',
 'B365A',
 'BFDH',
 'BFDD',
 'BFDA',
 'BVH',
 'BVD',
 'BVA',
 'BWH',
 'BWD',
 'BWA',
 'PPH',
 'PPD',
 'PPA',
 'SKBH',
 'SKBD',
 'SKBA',
 'MaxH',
 'MaxD',
 'MaxA',
 'AvgH',
 'AvgD',
 'AvgA',
 'BFEH',
 'BFED',
 'BFEA',
 'B365>2.5',
 'B365<2.5',
 'Max>2.5',
 'Max<2.5',
 'Avg>2.5',
 'Avg<2.5',
 'BFE>2.5',
 'BFE<2.5',
 'AHh',
 'B365AHH',
 'B365AHA',
 'MaxAHH',
 'MaxAHA',
 'AvgAHH',
 'AvgAHA',
 'BFEAHH',
 'BFEAHA',
 'B365CH',
 'B365CD',
 'B365CA',
 'BFDCH',
 'BFDCD',
 'BFDCA',
 'BVCH',
 'BVCD',
 'BVCA',
 'BWCH',
 'BWCD',
 'BWCA',
 'PPCH',
 'PPCD',
 'PPCA',
 'SKBCH',
 'SKBCD',
 'SKBCA',
 'MaxCH',
 'MaxCD',
 'MaxCA',
 'AvgCH',
 'AvgCD',
 'AvgCA',
 'BFECH',
 'BFECD',
 'BFECA',
 'B365C>2.5',
 'B365C<2.5',
 'MaxC>2.5',
 'MaxC<2.5',
 'AvgC>2.5',
 'AvgC<2.5',
 'BFE

Data Source Notes

Primary Data Source
-football-data.co.uk

Historical Seasons:
-2024/25
-2025/26

Current Season:
-2026/27

Important Features:
-Goals
-Shots
-Shots on Target
-Betting Odds
-Expected Goals (Current Season)

Historical Team Ratings

In [9]:
historical_matches = pd.concat(
    [
        season_2024_25,
        season_2025_26
    ],
    ignore_index=True
)

In [10]:
historical_matches.shape

(760, 150)

In [11]:
current_matches = (
    season_2026_27.copy()
)

In [12]:
current_matches.shape

(60, 114)

In [13]:
matches = historical_matches[
    [
        'Date',
        'HomeTeam',
        'AwayTeam',
        'FTHG',
        'FTAG',
        'HS',
        'AS',
        'HST',
        'AST'
    ]
    ].copy()

In [14]:
matches.head()

,Date,HomeTeam,AwayTeam,FTHG,FTAG,HS,AS,HST,AST
0,16/08/2024,Man United,Fulham,1,0,14,10,5,2
1,17/08/2024,Ipswich,Liverpool,0,2,7,18,2,5
2,17/08/2024,Arsenal,Wolves,2,0,18,9,6,3
3,17/08/2024,Everton,Brighton,0,3,9,10,1,5
4,17/08/2024,Newcastle,Southampton,1,0,3,19,1,4


In [15]:
teams = sorted(
    pd.concat(
        [
            matches['HomeTeam'],
            matches['AwayTeam']
        ]
    ).unique()
)

In [16]:
len(teams)

23

In [17]:
team_ratings = pd.DataFrame(
    {
        'team':teams
    }
)

In [18]:
goals_for = []

for team in teams:

    home_goals = matches.loc[
        matches['HomeTeam'] == team,
        'FTHG'
        ].sum()

    away_goals = matches.loc[
        matches['AwayTeam'] == team,
        'FTAG'
        ].sum()

    goals_for.append(
        home_goals + away_goals
    )

team_ratings['goals_for'] = goals_for

Goals Against

In [20]:
goals_against = []

for team in teams:

    home_conceded = matches.loc[
        matches['HomeTeam'] == team,
        'FTAG'
    ].sum()

    away_conceded = matches.loc[
        matches['AwayTeam'] == team,
        'FTHG'
    ].sum()

    goals_against.append(
        home_conceded + away_conceded
    )

team_ratings['goals_against'] = goals_against

In [23]:
team_ratings[
    [
        'team',
        'goals_for',
        'goals_against'
    ]
    ].head()

,team,goals_for,goals_against
0,Arsenal,140,61
1,Aston Villa,114,100
2,Bournemouth,116,100
3,Brentford,121,109
4,Brighton,118,105


Shots For

In [22]:
shots_for = []

for team in teams:

    home_shots = matches.loc[
        matches['HomeTeam'] == team,
        'HS'
        ].sum()

    away_shots = matches.loc[
        matches['AwayTeam'] == team,
        'AS'
        ].sum()

    shots_for.append(
        home_shots + away_shots
    )

team_ratings['shots_for'] = shots_for


In [24]:
team_ratings[
    [
        'team',
        'shots_for'
    ]
    ].head()

,team,shots_for
0,Arsenal,1099
1,Aston Villa,967
2,Bournemouth,1105
3,Brentford,846
4,Brighton,1028


Shots Against

In [25]:
shots_against = []

for team in teams:

    home_allowed = matches.loc[
        matches['HomeTeam'] == team,
        'AS'
        ].sum()

    away_allowed = matches.loc[
        matches['AwayTeam'] == team,
        'HS'
        ].sum()

    shots_against.append(
        home_allowed + away_allowed
    )

team_ratings['shots_against'] = shots_against

In [26]:
team_ratings[
    [
        'team',
        'shots_for',
        'shots_against'
    ]
    ].head()

,team,shots_for,shots_against
0,Arsenal,1099,672
1,Aston Villa,967,953
2,Bournemouth,1105,967
3,Brentford,846,1151
4,Brighton,1028,870


In [27]:
team_ratings.head()

,team,goals_for,goals_against,shots_for,shots_against
0,Arsenal,140,61,1099,672
1,Aston Villa,114,100,967,953
2,Bournemouth,116,100,1105,967
3,Brentford,121,109,846,1151
4,Brighton,118,105,1028,870


Attack Ratings

In [28]:
league_avg_goals_for = (
    team_ratings['goals_for']
    .mean()
)

league_avg_goals_for

np.float64(93.91304347826087)

In [30]:
team_ratings['attack_rating'] = (
    team_ratings['goals_for']
    /
    league_avg_goals_for
)

In [31]:
team_ratings[
    [
        'team',
        'goals_for',
        'attack_rating'
    ]
    ].sort_values(
        by='attack_rating',
        ascending=False
    ).head(10)

,team,goals_for,attack_rating
13,Liverpool,149,1.586574
14,Man City,149,1.586574
0,Arsenal,140,1.490741
6,Chelsea,122,1.299074
16,Newcastle,121,1.288426
3,Brentford,121,1.288426
4,Brighton,118,1.256481
2,Bournemouth,116,1.235185
1,Aston Villa,114,1.213889
15,Man United,113,1.203241


Defense Rating

In [32]:
league_avg_goals_against = (
    team_ratings['goals_against']
    .mean()
)

league_avg_goals_against

np.float64(93.91304347826087)

In [33]:
team_ratings['defense_rating'] = (
    team_ratings['goals_against']
    /
    league_avg_goals_against
)

In [34]:
team_ratings[
    [
        'team',
        'goals_against',
        'defense_rating'
    ]
    ].sort_values(
        by='defense_rating',
        ascending=True
    ).head(10)

,team,goals_against,defense_rating
19,Sunderland,48,0.511111
11,Leeds,56,0.596296
0,Arsenal,61,0.649537
5,Burnley,75,0.798611
14,Man City,79,0.841204
12,Leicester,80,0.851852
10,Ipswich,82,0.873148
18,Southampton,86,0.915741
8,Everton,94,1.000926
13,Liverpool,94,1.000926


Matches Played

In [35]:
matches_played = []

for team in teams:

    home_matches = (
        matches['HomeTeam'] == team
    ).sum()

    away_matches = (
        matches['AwayTeam'] == team
    ).sum()

    matches_played.append(
        home_matches + away_matches
    )

team_ratings['matches_played'] = matches_played

In [36]:
team_ratings[
    [
        'team',
        'matches_played'
    ]
    ].sort_values(
        by='matches_played',
        ascending=False
    )

,team,matches_played
0,Arsenal,76
1,Aston Villa,76
2,Bournemouth,76
3,Brentford,76
4,Brighton,76
6,Chelsea,76
7,Crystal Palace,76
22,Wolves,76
8,Everton,76
9,Fulham,76


In [37]:
team_ratings['goals_for_per_match'] = (
    team_ratings['goals_for']
    /
    team_ratings['matches_played']
)

team_ratings['goals_against_per_match'] = (
    team_ratings['goals_against']
    /
    team_ratings['matches_played']
)

In [38]:
team_ratings[
    [
        'team',
        'goals_for_per_match',
        'goals_against_per_match'
    ]
    ]

,team,goals_for_per_match,goals_against_per_match
0,Arsenal,1.842105,0.802632
1,Aston Villa,1.500000,1.315789
2,Bournemouth,1.526316,1.315789
3,Brentford,1.592105,1.434211
4,Brighton,1.552632,1.381579
5,Burnley,1.000000,1.973684
6,Chelsea,1.605263,1.250000
7,Crystal Palace,1.210526,1.342105
8,Everton,1.171053,1.236842
9,Fulham,1.328947,1.381579


In [40]:
league_avg_gf_pm = (
    team_ratings['goals_for_per_match']
    .mean()
)

team_ratings['attack_rating'] = (
    team_ratings['goals_for_per_match']
    /
    league_avg_gf_pm
)

In [41]:
league_avg_ga_pm = (
    team_ratings['goals_against_per_match']
    .mean()
)

team_ratings['defense_rating'] = (
    team_ratings['goals_against_per_match']
    /
    league_avg_ga_pm
)

In [42]:
team_ratings[
    [
        'team',
        'matches_played',
        'attack_rating',
        'defense_rating'
    ]
    ].sort_values(
        by='attack_rating',
        ascending=False
    ).head(10)


,team,matches_played,attack_rating,defense_rating
13,Liverpool,76,1.437500,0.835717
14,Man City,76,1.437500,0.702358
0,Arsenal,76,1.350671,0.542327
6,Chelsea,76,1.177013,0.844608
16,Newcastle,76,1.167366,0.906842
3,Brentford,76,1.167366,0.969076
4,Brighton,76,1.138423,0.933514
2,Bournemouth,76,1.119128,0.889061
1,Aston Villa,76,1.099832,0.889061
15,Man United,76,1.090185,0.924623


Shot Creation Rating

In [43]:
team_ratings['shots_for_per_match'] = (
    team_ratings['shots_for']
    /
    team_ratings['matches_played']
)

In [44]:
league_avg_shots_pm = (
    team_ratings['shots_for_per_match']
    .mean()
)

league_avg_shots_pm

np.float64(12.374141876430206)

In [47]:
team_ratings['shot_creation_rating'] = (
    team_ratings['shots_for_per_match']
    /
    league_avg_shots_pm
)

In [48]:
team_ratings[
    [
        'team',
        'shots_for_per_match',
        'shot_creation_rating'
    ]
    ].sort_values(
        by='shot_creation_rating',
        ascending=False
    ).head(10)

,team,shots_for_per_match,shot_creation_rating
13,Liverpool,16.263158,1.314286
14,Man City,15.815789,1.278132
15,Man United,14.776316,1.194129
6,Chelsea,14.552632,1.176052
2,Bournemouth,14.539474,1.174988
0,Arsenal,14.460526,1.168608
4,Brighton,13.526316,1.093111
16,Newcastle,13.368421,1.080351
9,Fulham,13.184211,1.065465
1,Aston Villa,12.723684,1.028248


Shot Prevention Rating

In [49]:
team_ratings['shots_against_per_match'] = (
    team_ratings['shots_against']
    /
    team_ratings['matches_played']
)

In [50]:
team_ratings[
    [
        'team',
        'shots_against_per_match'
    ]
    ].head()

,team,shots_against_per_match
0,Arsenal,8.842105
1,Aston Villa,12.539474
2,Bournemouth,12.723684
3,Brentford,15.144737
4,Brighton,11.447368


In [51]:
league_avg_shots_against_pm = (
    team_ratings['shots_against_per_match']
    .mean()
)

In [53]:
team_ratings['shot_prevention_rating'] = (
    team_ratings['shots_against_per_match']
    /
    league_avg_shots_against_pm
)

In [54]:
team_ratings[
    [
        'team',
        'shots_against_per_match',
        'shot_prevention_rating'
    ]
    ].sort_values(
        by='shot_prevention_rating',
        ascending=True
    ).head(10)

,team,shots_against_per_match,shot_prevention_rating
0,Arsenal,8.842105,0.673993
14,Man City,9.526316,0.726147
6,Chelsea,10.697368,0.815411
13,Liverpool,10.815789,0.824437
15,Man United,11.263158,0.858538
4,Brighton,11.447368,0.872580
9,Fulham,11.578947,0.882609
7,Crystal Palace,12.223684,0.931755
16,Newcastle,12.407895,0.945796
11,Leeds,12.526316,0.954823


Overall Power Ranking

In [55]:
team_ratings['defensive_strength'] = (
    (
        1 / team_ratings['defense_rating']
    )
    +
    (
        1 / team_ratings['shot_prevention_rating']
    )
)

In [56]:
team_ratings['offensive_strength'] = (
    team_ratings['attack_rating']
    +
    team_ratings['shot_creation_rating']
)

In [57]:
team_ratings['power_rating'] = (
    team_ratings['offensive_strength']
    +
    team_ratings['defensive_strength']
)

In [60]:
team_ratings[
    [
        'team',
        'power_rating'
    ]
    ].sort_values(
        by='power_rating',
        ascending=False
    ).head(15)

,team,power_rating
0,Arsenal,5.846881
14,Man City,5.516540
13,Liverpool,5.161311
6,Chelsea,4.763423
15,Man United,4.530605
2,Bournemouth,4.449967
4,Brighton,4.448783
16,Newcastle,4.407755
1,Aston Villa,4.299078
9,Fulham,4.244103


EPL Power Rankings

In [61]:
power_rankings = (
    team_ratings[
        [
            'team',
            'power_rating'
        ]
        ]
    .sort_values(
        by='power_rating',
        ascending=False
    )
    .reset_index(drop=True)
)

In [62]:
power_rankings.index = (
    power_rankings.index + 1
)

power_rankings.index.name = 'rank'

power_rankings.head(20)

,team,power_rating
rank,,
1,Arsenal,5.846881
2,Man City,5.516540
3,Liverpool,5.161311
4,Chelsea,4.763423
5,Man United,4.530605
6,Bournemouth,4.449967
7,Brighton,4.448783
8,Newcastle,4.407755
9,Aston Villa,4.299078


In [63]:
power_rankings['power_rating'] = (
    power_rankings['power_rating']
    .round(3)
)

In [64]:
power_rankings.head(20)

,team,power_rating
rank,,
1,Arsenal,5.847
2,Man City,5.517
3,Liverpool,5.161
4,Chelsea,4.763
5,Man United,4.531
6,Bournemouth,4.450
7,Brighton,4.449
8,Newcastle,4.408
9,Aston Villa,4.299


In [66]:
power_rankings.to_csv(
    '../outputs/epl_power_rankings.csv'
)

Results

Top 10 EPL Power Rankings

1. Arsenal
2. Manchester City
3. Liverpool
4. Chelsea
5. Manchester United
6. Bournemouth
7. Brighton
8. Newcastle
9. Aston Villa
10. Fulham

Methodology:

-Attack Rating
-Defense Rating
-Shot Creation Rating
-Shot Prevention Rating

Power Rating = offensive strength + defensive strength

Limitations

Version 1 does not include:

-Expected Goals (xG)
-Current Form
-Home Advantage
-Match Predictions
-Season Simulations

Future versions may incorporate these features

Conclusions

The model identifies Arsenal, Manchester City, and Liverpool as the strongest clubs in the Premier Leaugue based on historical goal production and shot metrics. 

Future iterations will incorporate current-season performance and expected goal data